# 02 DPO

Purpose: build verifier-backed preference pairs and train/evaluate the FinChain DPO continuation used in this run.

Expected inputs: selected SFT checkpoint from `shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected`, generated DPO pair shards, and FinChain validation/test splits.

Expected outputs: DPO candidate checkpoints, validation-selected model, eval summaries, and candidate metadata for `shannan-liu1/qwen25-1p5b-finchain-v2-dpo-selected` / `shannan-liu1/qwen25-1p5b-finchain-v2-dpo-candidates`.

HF status: see `docs/hf_checkpoints.md` before pushing.


## Before you run a single cell in this notebook - terminal pre-flight

This notebook runs offline Direct Preference Optimization (DPO) on verifier-built FinChain chosen/rejected pairs. Run this terminal block first on a fresh GPU environment. It pins the CUDA/PyTorch stack before NCCL work, keeps HF cache on the persistent volume, requires the FinChain v2 template-disjoint JSONLs and manifest under the local data path, and logs into W&B/Hugging Face before training starts. If you are restarting a stopped environment or recovering from a package failure, rerun this block before resuming training; do not jump straight to an Accelerate launch.

```bash
cd /workspace
test -d finpost || git clone https://github.com/shannan-liu1/finpost.git
cd /workspace/finpost
git checkout main
git pull --ff-only

# Install project deps, then fail fast on CUDA/NCCL drift. If the guard fails,
# the repair script removes CUDA 13 pip packages and reinstalls the A40-safe
# Torch CUDA 12.4 stack:
#   torch==2.6.0+cu124
# The repair script also removes optional torchvision/torchaudio wheels;
# finpost does not use them, and broken optional wheels can make transformers imports fail.
python -m pip install -e ".[dev,rlvr,chaineval]"
nvidia-smi || true
bash scripts/repair_cuda_stack.sh
python -m pip install -e ".[dev,rlvr,chaineval]"
python scripts/check_cuda_stack.py

# Persistent cache + FinChain split paths. The public repo does not
# track these JSONLs; generate or place them under data/finchain_v2_template_disjoint
# before the paid GPU run.
export HF_HOME=/workspace/hf-cache
export WANDB_MODE=${WANDB_MODE:-online}
export WANDB_PROJECT=finpost-finchain-dpo
mkdir -p /workspace/data/finchain_v2_template_disjoint /workspace/hf-cache
for f in train.jsonl validation.jsonl test.jsonl manifest.json; do
  test -f "data/finchain_v2_template_disjoint/$f" || { echo "Missing data/finchain_v2_template_disjoint/$f; generate or place the FinChain v2 splits before training." >&2; exit 1; }
  cp -n "data/finchain_v2_template_disjoint/$f" /workspace/data/finchain_v2_template_disjoint/
done
export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain_v2_template_disjoint/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain_v2_template_disjoint/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain_v2_template_disjoint/test.jsonl
python scripts/audit_finchain_template_disjoint_manifest.py --data-dir data/finchain_v2_template_disjoint --out artifacts/preflight/finchain_v2_manifest_audit.json
python scripts/gpu_preflight.py --out artifacts/preflight/preflight_report.json --timeout-sec 900

# Auth. `wandb status` should show your username. `huggingface-cli whoami`
# should succeed if you plan to push checkpoints or access private/gated repos.
wandb login
wandb status
huggingface-cli login
huggingface-cli whoami

# Pre-download model snapshots so an Accelerate launch does not look hung while
# it is only fetching weights.
HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py Qwen/Qwen2.5-0.5B Qwen/Qwen2.5-1.5B
```

Distributed rule of thumb: record distributed training only for cells that actually use `accelerate launch --num_processes 2` or explicit pair-generation sharding. `NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1` is the safe 2x A40 starting point; remove those only after a small distributed canary passes without the NCCL/CUDA-driver error.


# FinChain DPO GPU Notebook

Runs the repo-native DPO trainer (`finpost.training.dpo_train`) on FinChain preference pairs. DPO is the fixed-offline-pair baseline in the four-method comparison (SFT, DPO, GRPO, GKD). Multi-GPU full training uses `scripts/train_finchain_trl_dpo.py` with `accelerate launch`.

Prerequisite: an HF-format policy checkpoint at `results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected`. This public run uses the validation-selected SFT checkpoint produced by the SFT notebook (`notebooks/01_sft_ablation.ipynb`).

## Step 1 - Sanity-check the environment

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import os
import platform
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/workspace/finpost") if Path("/workspace/finpost").exists() else PROJECT_ROOT
os.chdir(PROJECT_ROOT)

RESULTS_DIR = PROJECT_ROOT / "results" / "finchain_dpo"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def progress(title, detail=None):
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {title}")
    if detail:
        print(detail)


def run_cmd(cmd, *, check=False):
    progress("running command", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {cmd}")
    return completed


def append_cost_event(stage, **payload):
    path = RESULTS_DIR / "cost_ledger.jsonl"
    row = {"stage": stage, "time": time.strftime("%Y-%m-%dT%H:%M:%S"), **payload}
    with path.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(row, sort_keys=True) + "\n")
    print(json.dumps(row, indent=2, sort_keys=True))
    return row


progress("project root", str(PROJECT_ROOT))
progress("python", sys.version.split()[0])
progress("platform", platform.platform())

## Distributed launch preflight

Run this after the setup cell and before any multi-GPU command. If it reports fewer than 2 CUDA devices, stay on the single-GPU path. See `notebooks/01_sft_ablation.ipynb` for the full environment checklist.

In [ ]:
progress("distributed launch preflight")
try:
    import accelerate
    print("accelerate:", accelerate.__version__)
except Exception as exc:
    print("accelerate import failed:", repr(exc))

try:
    import torch
    print("cuda devices:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch cuda check failed:", repr(exc))

for key in ["CUDA_VISIBLE_DEVICES", "WORLD_SIZE", "LOCAL_RANK", "RANK"]:
    print(f"{key}={os.environ.get(key)}")

run_cmd("accelerate env")


## GPU setup guardrails

See `notebooks/01_sft_ablation.ipynb` for the full checklist. DPO-specific note: pair generation shards prompts with `--shard-id/--num-shards`, then TRL DPO uses `accelerate launch` for DDP training on the merged fixed-pair JSONL.

In [ ]:
import os
import shutil
from pathlib import Path

os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
os.environ.setdefault("WANDB_MODE", "online")
os.environ.setdefault("WANDB_PROJECT", "finpost-finchain-dpo")

runtime_data_dir = Path("/workspace/data/finchain_v2_template_disjoint")
repo_data_dir = PROJECT_ROOT / "data" / "finchain_v2_template_disjoint"
runtime_data_dir.mkdir(parents=True, exist_ok=True)

missing_inputs = []
for split in ["train", "validation", "test"]:
    source = repo_data_dir / f"{split}.jsonl"
    target = runtime_data_dir / f"{split}.jsonl"
    if not target.exists() and source.exists():
        shutil.copy2(source, target)
    os.environ[f"FINPOST_FINCHAIN_{split.upper()}_JSONL"] = str(target)
    if not target.exists():
        missing_inputs.append(str(target))

manifest_source = repo_data_dir / "manifest.json"
manifest_target = runtime_data_dir / "manifest.json"
if not manifest_target.exists() and manifest_source.exists():
    shutil.copy2(manifest_source, manifest_target)
if not manifest_target.exists():
    missing_inputs.append(str(manifest_target))

if missing_inputs:
    raise FileNotFoundError(
        "Missing FinChain v2 split JSONLs or manifest. Generate or place them under "
        "data/finchain_v2_template_disjoint/ before training: "
        + ", ".join(missing_inputs)
    )

progress("GPU CUDA/data guard")
run_cmd("python scripts/check_cuda_stack.py", check=True)

progress("DPO TRL dependency guard")
run_cmd("python scripts/train_finchain_trl_dpo.py --help", check=True)
run_cmd("""
python - <<'PY'
import argparse
from trl import DPOConfig
from scripts.train_finchain_trl_dpo import _config_kwargs

args = argparse.Namespace(
    output_dir="artifacts/preflight/dpo_config_smoke",
    max_steps=1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    beta=0.1,
    max_length=1536,
    max_prompt_length=768,
    save_steps=50,
    logging_steps=5,
    report_to="none",
    run_name="dpo-config-smoke",
    seed=42,
)
DPOConfig(**_config_kwargs(args, config_cls=DPOConfig))
print("DPOConfig OK")
PY
""", check=True)

progress("auth status")
run_cmd("wandb status", check=False)
run_cmd("huggingface-cli whoami", check=False)

for key in [
    "HF_HOME",
    "WANDB_MODE",
    "WANDB_PROJECT",
    "FINPOST_FINCHAIN_TRAIN_JSONL",
    "FINPOST_FINCHAIN_VALIDATION_JSONL",
    "FINPOST_FINCHAIN_TEST_JSONL",
]:
    print(f"{key}={os.environ.get(key)}")


## Step 1.5 - Hugging Face cache warmup

Run once per environment/volume. This downloads tokenizer/config/safetensors into `HF_HOME` without loading the model on GPU. It makes later stalls easier to diagnose: after this cell, a long pause is trainer setup or generation, not model download.


In [ ]:
HF_WARMUP_MODELS = ['Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-1.5B']
progress("HF cache warmup", " ".join(HF_WARMUP_MODELS))
warm_cmd = "HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py " + " ".join(HF_WARMUP_MODELS)
run_cmd(warm_cmd, check=True)


In [ ]:
progress("GPU preflight")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch check failed:", repr(exc))

run_cmd("nvidia-smi")

In [ ]:
# Confirm core modules import. If any fail, the install didn't take.
for module_name in [
    "finpost.training.dpo_train",
    "finpost.training.preference_data",
    "finpost.training.logprobs",
    "finpost.data.finchain_dataset",
    "finpost.evals.finchain_metrics",
]:
    try:
        importlib.import_module(module_name)
        print(module_name, "OK")
    except Exception as exc:
        print(module_name, "FAILED:", repr(exc))

from finpost.data.finchain_dataset import load_finchain, resolve_finchain_path

split_examples = {}
for split in ["train", "validation", "test"]:
    path = resolve_finchain_path(split)
    examples = load_finchain(split)
    split_examples[split] = examples
    print(split, "->", path, "exists=", path.exists(), "rows=", len(examples))

assert len(split_examples["validation"]) == 870, "expected FinChain validation split to contain 870 rows"
assert len(split_examples["test"]) == 870, "expected FinChain test split to contain 870 rows"
train_examples = split_examples["train"]
print("train examples:", len(train_examples))
print("validation examples:", len(split_examples["validation"]))
print("test examples:", len(split_examples["test"]))
print("first prompt id:", train_examples[0].id)
print("first gold answer:", train_examples[0].final_answer)


## Step 2 - Hyperparameters

The full run generates fresh FinChain preference pairs, trains TRL DPO to step 300, evaluates checkpoints 200, 250, and 300 on validation, and runs the validation-selected DPO checkpoint once against the SFT baseline on test.

In [ ]:
DPO_CONFIG = {
    # Pair generation and DPO training are decoupled: --model-checkpoint can
    # sample from any compatible HF-format causal LM. This run uses the
    # validation-selected SFT checkpoint as both pair source and policy/reference
    # initialization for a fair continuation baseline.
    "base_model_id": "Qwen/Qwen2.5-1.5B",
    "fast_canary_model_id": "Qwen/Qwen2.5-0.5B",
    "policy_hf_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected",
    "pair_source_checkpoint": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected",
    "dpo_config_yaml": "configs/finchain/dpo/finchain_qwen25_1_5b.yaml",
    # Fresh pair generation. This notebook resets pair_run_root before the
    # full rerun so old duplicate-prone pairs cannot leak into training.
    "pair_run_root": "results/finchain_pairs/v2_template_disjoint",
    "pair_samples_per_prompt": 8,
    "pair_max_new_tokens": 768,
    "pair_max_pairs_per_prompt": 1,
    "pair_generation_batch": 64,
    "pair_heldout_train_n": 2320,
    # DPO training: save every 50 steps with save_total_limit=3 in the TRL
    # adapter, so a 300-step run retains checkpoint-200/250/300.
    "dpo_save_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-dpo",
    "dpo_trl_save_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-dpo-trl-2gpu",
    "dpo_selected_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-dpo-selected",
    "dpo_max_steps": 300,
    "dpo_save_steps": 50,
    "dpo_eval_steps": [200, 250, 300],
    "dpo_beta": 0.1,
    "dpo_lr": 5.0e-6,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "wandb_project": "finpost-finchain-dpo",
    "run_name": "qwen25-1p5b-finchain-v2-dpo-300step-rerun",
    # HF repos to overwrite after validation/test succeeds.
    "hf_selected_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-dpo-selected",
    "hf_candidates_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-dpo-candidates",
    # Set true to remove stale local artifacts before rerun.
    "reset_outputs_before_rerun": True,
}
print(json.dumps(DPO_CONFIG, indent=2))


## Step 2.5 - Reset stale DPO artifacts for a fair rerun

This removes old local DPO pair, checkpoint, selected-checkpoint, and eval outputs before generating fresh pairs. It does not touch the configured policy/source checkpoint or Hugging Face repos. HF cleanup happens only in the final upload cells after validation/test succeeds.


In [ ]:
import shutil

if DPO_CONFIG["reset_outputs_before_rerun"]:
    reset_paths = [
        Path(DPO_CONFIG["pair_run_root"]),
        Path(DPO_CONFIG["dpo_save_dir"]),
        Path(DPO_CONFIG["dpo_trl_save_dir"]),
        Path(DPO_CONFIG["dpo_selected_dir"]),
        Path(f"{DPO_CONFIG['dpo_save_dir']}-hf-candidates"),
        PROJECT_ROOT / "results" / "evals" / "dpo_v2_template_disjoint",
    ]
    for reset_path in reset_paths:
        if reset_path.exists():
            print("removing stale artifact:", reset_path)
            shutil.rmtree(reset_path)
else:
    print("reset_outputs_before_rerun is False; keeping existing local DPO artifacts.")


## Step 3 - Fast canary pair-gen (0.5B, 32 prompts)

Catches: FinChain loader misconfigured, verifier regex broken, generation path broken, CUDA stack unstable. This deliberately uses `Qwen/Qwen2.5-0.5B` so the canary is cheap. The full pair-gen below uses the configured 1.5B pair-source checkpoint and is the artifact you would cite.


In [ ]:
canary_pair_dir = Path(DPO_CONFIG["pair_run_root"]) / "canary"
canary_model = DPO_CONFIG["fast_canary_model_id"]
canary_pair_cmd = (
    "HF_HOME=/workspace/hf-cache CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py "
    f"--model-checkpoint {canary_model} "
    "--sources finchain "
    f"--out-dir {canary_pair_dir} "
    "--heldout-train-n 32 "
    "--samples-per-prompt 2 "
    "--generation-batch-size 16 "
    "--max-new-tokens-finchain 128 "
    "--max-pairs-per-prompt 2 "
    "--allow-empty-pairs"
)
canary_start = time.perf_counter()
canary_result = run_cmd(canary_pair_cmd, check=False)
canary_elapsed = time.perf_counter() - canary_start
append_cost_event(
    "pair_gen_canary_fast_0p5b",
    elapsed_sec=round(canary_elapsed, 1),
    exit_code=canary_result.returncode,
    out_dir=str(canary_pair_dir),
    model=canary_model,
)
if canary_result.returncode != 0:
    raise RuntimeError("Pair-gen canary failed. Fix before proceeding.")


## Step 4 - Full FinChain pair generation

Default path: two-GPU pair generation when two CUDA devices are visible. Each GPU owns a deterministic prompt shard via `--shard-id / --num-shards`, then the merge step writes one merged `pairs.jsonl`. The fallback path is single-GPU pair generation only when the environment exposes fewer than two GPUs.


In [ ]:
import torch as _t

merged_pair_dir = Path(DPO_CONFIG["pair_run_root"]) / "merged"
gpu_count = _t.cuda.device_count() if _t.cuda.is_available() else 0
if gpu_count >= 2:
    pair_run = Path(DPO_CONFIG["pair_run_root"])
    pair_cmd = f"""
HF_HOME=/workspace/hf-cache CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py --model-checkpoint {DPO_CONFIG['pair_source_checkpoint']} --sources finchain --out-dir {pair_run}/shards/shard-00-of-02 --heldout-train-n {DPO_CONFIG['pair_heldout_train_n']} --samples-per-prompt {DPO_CONFIG['pair_samples_per_prompt']} --generation-batch-size {DPO_CONFIG['pair_generation_batch']} --max-new-tokens-finchain {DPO_CONFIG['pair_max_new_tokens']} --max-pairs-per-prompt {DPO_CONFIG['pair_max_pairs_per_prompt']} --shard-id 0 --num-shards 2 &
HF_HOME=/workspace/hf-cache CUDA_VISIBLE_DEVICES=1 python scripts/build_dpo_pairs.py --model-checkpoint {DPO_CONFIG['pair_source_checkpoint']} --sources finchain --out-dir {pair_run}/shards/shard-01-of-02 --heldout-train-n {DPO_CONFIG['pair_heldout_train_n']} --samples-per-prompt {DPO_CONFIG['pair_samples_per_prompt']} --generation-batch-size {DPO_CONFIG['pair_generation_batch']} --max-new-tokens-finchain {DPO_CONFIG['pair_max_new_tokens']} --max-pairs-per-prompt {DPO_CONFIG['pair_max_pairs_per_prompt']} --shard-id 1 --num-shards 2 &
wait
HF_HOME=/workspace/hf-cache python scripts/merge_dpo_pair_shards.py --shard-dirs {pair_run}/shards/shard-00-of-02 {pair_run}/shards/shard-01-of-02 --out-dir {merged_pair_dir} --quality-gate finchain-full
""".strip()
    cost_label = "pair_gen_full_two_gpu"
else:
    pair_cmd = (
        "HF_HOME=/workspace/hf-cache CUDA_VISIBLE_DEVICES=0 python scripts/build_dpo_pairs.py "
        f"--model-checkpoint {DPO_CONFIG['pair_source_checkpoint']} "
        "--sources finchain "
        f"--out-dir {merged_pair_dir} "
        f"--heldout-train-n {DPO_CONFIG['pair_heldout_train_n']} "
        f"--samples-per-prompt {DPO_CONFIG['pair_samples_per_prompt']} "
        f"--generation-batch-size {DPO_CONFIG['pair_generation_batch']} "
        f"--max-new-tokens-finchain {DPO_CONFIG['pair_max_new_tokens']} "
        f"--max-pairs-per-prompt {DPO_CONFIG['pair_max_pairs_per_prompt']} "
        "--quality-gate finchain-full"
    )
    cost_label = "pair_gen_full_single_gpu"
    print("WARNING: fewer than 2 CUDA devices visible; falling back to single-GPU pair generation.")

pair_start = time.perf_counter()
pair_result = run_cmd(pair_cmd, check=True)
pair_manifest_path = merged_pair_dir / "manifest.json"
pair_manifest = json.loads(pair_manifest_path.read_text(encoding="utf-8"))
pair_summary = pair_manifest.get("pair_summary", {})
pair_quality = pair_manifest.get("pair_quality", {})
quality_gate = pair_manifest.get("quality_gate") or {}
print("pair quality:")
print(json.dumps(pair_quality, indent=2, sort_keys=True))
print("pair review sample:", pair_manifest.get("pair_review_sample_path"))
if not quality_gate.get("passed", False):
    raise RuntimeError(f"Fresh DPO pairs failed quality gate: {quality_gate}")
if pair_quality.get("pair_count", 0) <= 0:
    raise RuntimeError("Fresh DPO pair generation produced zero pairs.")
if pair_quality.get("duplicate_exact_pair_count", 0) != 0:
    raise RuntimeError(f"Fresh DPO pairs still contain exact duplicates: {pair_quality}")
append_cost_event(
    cost_label,
    elapsed_sec=round(time.perf_counter() - pair_start, 1),
    exit_code=pair_result.returncode,
    out_dir=str(merged_pair_dir),
    pair_quality=pair_quality,
)


## Step 5 - Fast DPO canary (0.5B, canary pair set)

Catches: TRL/repo-native DPO objective path broken, NaN loss, tokenizer cache issues, pad/eos regressions. This writes a temporary canary YAML under `artifacts/generated_configs/` pointed at the 0.5B canary pairs, so the 1.5B policy checkpoint is not loaded for a plumbing check.


In [ ]:
import yaml

generated_config_dir = Path("artifacts/generated_configs/dpo")
generated_config_dir.mkdir(parents=True, exist_ok=True)
canary_dpo_yaml = generated_config_dir / "finchain_qwen25_0_5b_canary.generated.yaml"
canary_cfg = yaml.safe_load(Path(DPO_CONFIG["dpo_config_yaml"]).read_text(encoding="utf-8"))
canary_cfg["model"]["policy_checkpoint"] = DPO_CONFIG["fast_canary_model_id"]
canary_cfg["model"]["reference_checkpoint"] = DPO_CONFIG["fast_canary_model_id"]
canary_cfg["data"]["pairs_path"] = str(canary_pair_dir / "pairs.jsonl")
canary_cfg["data"]["manifest_path"] = str(canary_pair_dir / "manifest.json")
canary_cfg["data"]["tokenized_cache_path"] = str(canary_pair_dir / "pairs.tokenized.pt")
canary_cfg["data"]["reference_logps_cache_path"] = str(canary_pair_dir / "pairs.reference-logps.pt")
canary_cfg["data"]["rebuild_tokenized_cache"] = True
canary_cfg["training"]["max_steps"] = 20
canary_cfg["training"]["warmup_steps"] = 2
canary_cfg["training"]["per_device_pair_batch_size"] = 1
canary_cfg["training"]["grad_accum_steps"] = 2
canary_cfg["checkpointing"]["save_dir"] = str(Path(DPO_CONFIG["dpo_save_dir"] + "-canary-0p5b"))
canary_cfg["logging"]["run_name"] = DPO_CONFIG["run_name"] + "-canary-0p5b"
canary_dpo_yaml.write_text(yaml.safe_dump(canary_cfg, sort_keys=False), encoding="utf-8")
print("wrote", canary_dpo_yaml)

dpo_canary_cmd = (
    "WANDB_MODE=offline HF_HOME=/workspace/hf-cache python -m finpost.training.dpo_train "
    f"--config {canary_dpo_yaml} "
    "--device cuda"
)
canary_start = time.perf_counter()
canary_result = run_cmd(dpo_canary_cmd, check=False)
append_cost_event(
    "dpo_canary_fast_0p5b",
    elapsed_sec=round(time.perf_counter() - canary_start, 1),
    exit_code=canary_result.returncode,
)
if canary_result.returncode != 0:
    raise RuntimeError("DPO canary failed. Fix before the full DPO run.")


## Step 6 - Full DPO training run

The TRL adapter uses `save_total_limit=3`. With `max_steps=300` and `save_steps=50`, the expected retained checkpoint set is `checkpoint-200`, `checkpoint-250`, and `checkpoint-300`. The next cell refuses to evaluate anything else.


In [ ]:
import torch as _t

pair_path = merged_pair_dir / "pairs.jsonl"
if not pair_path.exists():
    raise FileNotFoundError(f"missing fresh DPO pairs: {pair_path}")

gpu_count = _t.cuda.device_count() if _t.cuda.is_available() else 0
common_dpo_args = (
    "scripts/train_finchain_trl_dpo.py "
    f"--pairs-path {pair_path} "
    f"--model {DPO_CONFIG['policy_hf_dir']} "
    f"--max-steps {DPO_CONFIG['dpo_max_steps']} "
    f"--save-steps {DPO_CONFIG['dpo_save_steps']} "
    f"--learning-rate {DPO_CONFIG['dpo_lr']} "
    f"--beta {DPO_CONFIG['dpo_beta']} "
    f"--per-device-train-batch-size {DPO_CONFIG['per_device_train_batch_size']} "
    f"--gradient-accumulation-steps {DPO_CONFIG['gradient_accumulation_steps']} "
    f"--output-dir {DPO_CONFIG['dpo_trl_save_dir']} "
    f"--report-to wandb --run-name {DPO_CONFIG['run_name']}"
)
if gpu_count >= 2:
    dpo_full_cmd = (
        "HF_HOME=/workspace/hf-cache NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 "
        "accelerate launch --num_processes 2 --mixed_precision bf16 "
        + common_dpo_args
    )
    cost_label = "dpo_full_trl_two_gpu_300step"
else:
    dpo_full_cmd = "HF_HOME=/workspace/hf-cache python " + common_dpo_args
    cost_label = "dpo_full_trl_single_gpu_300step"
    print("WARNING: fewer than 2 CUDA devices visible; falling back to single-GPU TRL DPO.")

full_start = time.perf_counter()
full_result = run_cmd(dpo_full_cmd, check=True)
append_cost_event(
    cost_label,
    elapsed_sec=round(time.perf_counter() - full_start, 1),
    exit_code=full_result.returncode,
)


## Step 7 - Verify checkpoints landed

In [ ]:
# TRL checkpoints are already HF-format. For this rerun, evaluate only the
# requested validation candidates: checkpoint-200, checkpoint-250, checkpoint-300.
trl_save_dir = Path(DPO_CONFIG["dpo_trl_save_dir"])
if not trl_save_dir.exists():
    raise FileNotFoundError(f"TRL DPO output directory does not exist: {trl_save_dir}")

all_checkpoint_dirs = sorted(trl_save_dir.glob("checkpoint-*"))
print("all checkpoint dirs:", [path.name for path in all_checkpoint_dirs])

dpo_candidate_dirs = {}
for step in DPO_CONFIG["dpo_eval_steps"]:
    path = trl_save_dir / f"checkpoint-{step}"
    model_files = list(path.glob("*.safetensors")) + list(path.glob("pytorch_model*.bin"))
    trainer_state = path / "trainer_state.json"
    print(path.name, "exists=", path.exists(), "weights=", bool(model_files), "trainer_state=", trainer_state.exists())
    if not path.exists() or not model_files:
        raise RuntimeError(f"missing or incomplete DPO checkpoint for validation: {path}")
    dpo_candidate_dirs[f"dpo_checkpoint_{step}"] = path

if set(dpo_candidate_dirs) != {f"dpo_checkpoint_{step}" for step in DPO_CONFIG["dpo_eval_steps"]}:
    raise RuntimeError(f"candidate set mismatch: {list(dpo_candidate_dirs)}")
print("DPO validation candidates:", list(dpo_candidate_dirs))


## Step 8 - Select DPO on validation, then evaluate test once

Validation chooses among exactly `checkpoint-200`, `checkpoint-250`, and `checkpoint-300`. Test remains untouched until after that selection and compares only SFT versus the selected DPO checkpoint.


In [ ]:
import shutil
import torch as _t

EVAL_ROOT = PROJECT_ROOT / "results" / "evals" / "dpo_v2_template_disjoint"
VALIDATION_OUT_DIR = EVAL_ROOT / "validation_selection"
TEST_OUT_DIR = EVAL_ROOT / "test_selected_chaineval"
EVAL_CONFIG = {"parallelism": "examples", "batch_size_finchain": 32}
gpus_arg = "0 1" if _t.cuda.is_available() and _t.cuda.device_count() > 1 else "0"
candidate_args = " ".join(f"{label}={path}" for label, path in dpo_candidate_dirs.items())

run_cmd(
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints {candidate_args} --finchain-split validation --n 870 "
    f"--out-dir {VALIDATION_OUT_DIR} --batch-size-finchain {EVAL_CONFIG['batch_size_finchain']}",
    check=True,
)
validation_summaries = {
    label: json.loads((VALIDATION_OUT_DIR / label / "accuracy_summary.json").read_text(encoding="utf-8"))[0]
    for label in dpo_candidate_dirs
}


def checkpoint_step(label: str) -> int:
    return int(label.rsplit("_", 1)[-1])


selected_label = sorted(
    validation_summaries,
    key=lambda label: (
        -validation_summaries[label]["accuracy"],
        -validation_summaries[label].get("parse_success_rate", 0.0),
        checkpoint_step(label),
    ),
)[0]
dpo_selected_dir = Path(DPO_CONFIG["dpo_selected_dir"])
if dpo_selected_dir.exists():
    shutil.rmtree(dpo_selected_dir)
shutil.copytree(dpo_candidate_dirs[selected_label], dpo_selected_dir)
print("selected on validation:", selected_label, dpo_selected_dir)
print(json.dumps(validation_summaries, indent=2, sort_keys=True))

selection_metadata = {
    "selection_split": "validation",
    "final_eval_split": "test",
    "selected_checkpoint": str(dpo_selected_dir),
    "selected_label": selected_label,
    "candidate_labels": list(dpo_candidate_dirs),
    "selection_metric": "accuracy",
    "tie_breaker": "parse_success_rate_then_earliest_step",
    "test_touched_before_selection": False,
}
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
(EVAL_ROOT / "selection_metadata.json").write_text(
    json.dumps(selection_metadata, indent=2, sort_keys=True),
    encoding="utf-8",
)

run_cmd(
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints sft={DPO_CONFIG['policy_hf_dir']} dpo_selected={dpo_selected_dir} "
    f"--finchain-split test --n 870 --out-dir {TEST_OUT_DIR} "
    f"--batch-size-finchain {EVAL_CONFIG['batch_size_finchain']} --enable-chaineval",
    check=True,
)
append_cost_event("dpo_v2_selected_test_chaineval", selection=selected_label)


## Step 9 - Headline numbers

The DPO checkpoint is selected from validation only. Report the `TEST_OUT_DIR`
comparison as the untouched result: `accuracy` is final-answer correctness,
and ChainEval adds chain precision, recall, F1, and alignment diagnostics.


In [ ]:
BASELINE_AND_SELECTED_LABELS = ["sft", "dpo_selected"]
HEADLINE_METRICS = ["n", "accuracy", "parse_success_rate", "step_recall", "step_precision", "step_f1"]


def compact_metrics(summary):
    return {metric: summary.get(metric) for metric in HEADLINE_METRICS if metric in summary}


test_summaries = {}
for name in BASELINE_AND_SELECTED_LABELS:
    summary_path = TEST_OUT_DIR / name / "accuracy_summary.json"
    test_summaries[name] = json.loads(summary_path.read_text(encoding="utf-8"))[0]
print("DPO validation selection:")
print(json.dumps(validation_summaries, indent=2, sort_keys=True))
print("DPO untouched test summary metrics:")
print(json.dumps({name: compact_metrics(summary) for name, summary in test_summaries.items()}, indent=2, sort_keys=True))
print("DPO untouched test full ChainEval summaries:")
print(json.dumps(test_summaries, indent=2, sort_keys=True))


## Push the selected DPO checkpoint to HF Hub

Run this only after validation selection, untouched test evaluation, and summary metrics look sane. This cell clears stale files in the selected-checkpoint repo before uploading the new validation-selected checkpoint.


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

api = HfApi()
selected_repo_id = DPO_CONFIG["hf_selected_repo"]
folder_to_push = Path(dpo_selected_dir)
if not folder_to_push.exists():
    raise FileNotFoundError(f"checkpoint folder does not exist: {folder_to_push}")

api.create_repo(selected_repo_id, exist_ok=True, private=False)
existing_files = [
    path
    for path in api.list_repo_files(selected_repo_id, repo_type="model")
    if path != ".gitattributes"
]
for repo_file in existing_files:
    try:
        api.delete_file(
            path_in_repo=repo_file,
            repo_id=selected_repo_id,
            repo_type="model",
            commit_message=f"Remove stale selected DPO file {repo_file}",
        )
    except HfHubHTTPError as exc:
        print("delete skipped", repo_file, repr(exc))

api.upload_folder(
    folder_path=str(folder_to_push),
    repo_id=selected_repo_id,
    repo_type="model",
    commit_message="FinChain v2 DPO 300-step validation-selected HF checkpoint",
)
print(f"pushed {folder_to_push} to https://huggingface.co/{selected_repo_id}")
append_cost_event("hf_push_dpo_v2_selected", repo_id=selected_repo_id, folder=str(folder_to_push))


## Push DPO candidates and eval artifacts to HF Hub

This resets the candidates/evals repo paths that this notebook owns, then uploads only `checkpoint-200`, `checkpoint-250`, `checkpoint-300`, and `evals/dpo_v2_template_disjoint`. This avoids stale old candidates such as 400/450/500 lingering in the Hub repo.


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

api = HfApi()
candidates_repo_id = DPO_CONFIG["hf_candidates_repo"]
api.create_repo(candidates_repo_id, exist_ok=True, private=False)

for folder in [
    "checkpoint-200",
    "checkpoint-250",
    "checkpoint-300",
    "checkpoint-400",
    "checkpoint-450",
    "checkpoint-500",
    "final",
    "evals",
]:
    try:
        api.delete_folder(
            path_in_repo=folder,
            repo_id=candidates_repo_id,
            repo_type="model",
            commit_message=f"Remove stale DPO candidate path {folder}",
        )
        print("deleted stale HF path:", folder)
    except HfHubHTTPError as exc:
        print("delete skipped", folder, repr(exc))

for step in DPO_CONFIG["dpo_eval_steps"]:
    path = dpo_candidate_dirs[f"dpo_checkpoint_{step}"]
    api.upload_folder(
        folder_path=str(path),
        repo_id=candidates_repo_id,
        repo_type="model",
        path_in_repo=path.name,
        commit_message=f"Upload DPO 300-step candidate {path.name}",
    )

api.upload_folder(
    folder_path=str(EVAL_ROOT),
    repo_id=candidates_repo_id,
    repo_type="model",
    path_in_repo="evals/dpo_v2_template_disjoint",
    commit_message="Upload DPO 300-step validation/test eval artifacts",
)

print(f"uploaded DPO candidates/evals to https://huggingface.co/{candidates_repo_id}")
append_cost_event(
    "hf_push_dpo_v2_candidates_evals",
    repo_id=candidates_repo_id,
    candidates=list(dpo_candidate_dirs),
    eval_root=str(EVAL_ROOT),
)


## Final - Stop the instance

GPU rentals can charge by the hour. After the final eval artifact is saved and no further cells are queued, stop the instance through your provider console or CLI.